Replication of Anthropic's Emotion Vector Findings inside Gemma 4 E2B

In [1]:
# Core Machine Learning & TPU Support
%pip install torch torch_xla[tpu] -f https://storage.googleapis.com/tpu-pytorch/wheels/tpuvm/torch_xla-2.1-cp310-cp310-linux_x86_64.whl
%pip install transformers==5.5.0 accelerate

# Interpretability & Visualization
%pip install plotly kaleido pandas scikit-learn huggingface-hub

Looking in links: https://storage.googleapis.com/tpu-pytorch/wheels/tpuvm/torch_xla-2.1-cp310-cp310-linux_x86_64.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 126.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 6.1 MB/s eta 0:00:00


In [2]:
import time
import json
import os
import gc
import zipfile
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import plotly.express as px
from typing import List, Dict
from transformers import AutoModelForCausalLM, AutoTokenizer
from accelerate import Accelerator
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from google.colab import files

# Constant values for the environment
kModelIdx = "google/gemma-4-E2B-it"
kOutDir = "./research_data"

# Global variables for the Collab refactor
gAccelerator = None
gDevice = None
gTokenizer = None
gModel = None
gTargetLayer = None # Layer 24 has consistent emotion classifications
gStoryFile = None
gEmotionLibrary: Dict[str, torch.Tensor] = None
gNeutralVectors: List[torch.Tensor] = None

In [3]:
kModelIdx = "openai-community/gpt2-medium"

In [3]:
kModelIdx = "google/gemma-4-E2B"

In [4]:
# @title
# Neutral prompt list from rain1955/emotion-vector-replication extendend with Google Gemini 3 Fast (04/09/2026)
neutralPrompts = [
  # prompts originating from rain1955/emotion-vector-replication
  "The meeting is scheduled for 3pm tomorrow.",
  "Please find the attached document.",
  "The temperature today is 22 degrees Celsius.",
  "The project deadline has been moved to next Friday.",
  "The store is located on the corner of Main Street.",
  "Chapter 3 discusses the economic implications.",
  "The software update includes several bug fixes.",
  "The report contains data from the past quarter.",
  "The committee will review the proposal next week.",
  "The library opens at 9am on weekdays.",
  # prompts generated with Google Gemini 3 Fast (04/09/2026)
  "The itinerary for the conference has been finalized.",
  "Data collection will commence at the beginning of the month.",
  "The user manual provides instructions for hardware setup.",
  "Standard procedure requires a signed authorization form.",
  "The server maintenance is performed every Sunday night.",
  "Historical records are stored in the basement archive.",
  "The chemical reaction occurs at room temperature.",
  "Please submit your expenses by the end of the day.",
  "The laboratory results will be available in 48 hours.",
  "The publication follows a strict peer-review process.",
  "The office is situated on the fourth floor of the building.",
  "Annual inspections are mandatory for all equipment.",
  "The software license expires at the end of the calendar year.",
  "The lecture series covers fundamental principles of physics.",
  "The supply chain manager coordinates all logistics.",
  "The router configuration remains unchanged since the last update.",
  "Participant feedback is collected through an anonymous survey.",
  "The contract specifies the terms and conditions of employment.",
  "Geological surveys indicate a high concentration of minerals.",
  "The system generates a log file for every transaction.",
  "Traffic flow is monitored by automated sensors.",
  "The workshop focuses on improving technical documentation.",
  "The final audit confirmed the accuracy of the financial statements.",
  "The museum is closed for renovations until next month.",
  "Utility bills are calculated based on monthly consumption.",
  "The inventory count is updated every Tuesday morning.",
  "The parking garage remains open 24 hours a day.",
  "The application process requires a valid form of identification.",
  "The thermostat is set to maintain a constant temperature.",
  "The user interface supports three different language options.",
  "The flight departure is confirmed for gate 12B.",
  "The building specifications meet the current safety codes.",
  "The printer requires a replacement toner cartridge.",
  "The database backup was completed successfully at midnight.",
  "The street lights are programmed to activate at sunset.",
  "The employee handbook outlines the company's privacy policy.",
  "The water filtration system is inspected twice a year.",
  "The package dimensions must not exceed the standard limit.",
  "The conference call will be recorded for future reference.",
  "The technical support team is available via email."
]

In [5]:
# @title
# Emotion word list from rain1955/emotion-vector-replication
'''
emotionLabels = [
  "happy", "sad", "angry", "afraid", "calm",
  "desperate", "loving", "guilty", "surprised", "nervous",
  "proud", "inspired", "spiteful", "brooding", "playful",
  "anxious", "confused", "disgusted", "lonely", "hopeful"
]
'''
emotionLabels = [
    'calm', 'loving', 'sad', 'guilty', 'desperate', 'afraid', 'angry', 'surprised', 'happy'
]

# Topics from rain1955/emotion-vector-replication
storyTopics = [
    "a student preparing for an exam",
    "a chef cooking a meal for guests",
    "a parent watching their child play",
    "a soldier returning home",
    "an artist finishing a painting",
    "a driver stuck in traffic",
    "a doctor delivering news to a patient",
    "a traveler arriving in a new city",
    "a musician performing on stage",
    "a shopkeeper closing for the day",
]

In [6]:
emotionLabels = [
  "happy", "sad", "angry", "afraid", "calm",
  "desperate", "loving", "guilty", "surprised", "nervous",
  "proud", "inspired", "spiteful", "brooding", "playful",
  "anxious", "confused", "disgusted", "lonely", "hopeful"
]

In [7]:
def initialize():
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    print(f"[INIT] Initializing Research Orchestrator for {kModelIdx}...")
    gAccelerator = Accelerator()
    gDevice = gAccelerator.device
    gTokenizer = AutoTokenizer.from_pretrained(kModelIdx)
    gTokenizer.padding_side = "right" # gpt 2 setting
    if gTokenizer.pad_token is None:
        gTokenizer.pad_token = gTokenizer.eos_token
    gModel = AutoModelForCausalLM.from_pretrained(
        kModelIdx,
        torch_dtype=torch.bfloat16
    ).to(gDevice)
    gModel.eval()
    gEmotionLibrary = {}
    gNeutralVectors = []
    gTargetLayer = 24 # Layer 24 has consistent emotion classifications
    gStoryFile = os.path.join(kOutDir, "emotion_stories.json")
    print(f"[INIT] Model loaded. Target Layer: {gTargetLayer} | Device: {gDevice}")

def freeVRAM():
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gAccelerator.free_memory()

def normalizeVector(vector):
    vector = vector.view(-1)  # force 1D
    return vector / (vector.norm() + 1e-9)

def computeCosineSimilarity(vectorA, vectorB):
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    return F.cosine_similarity(vectorA.unsqueeze(0), vectorB.unsqueeze(0)).item()

In [8]:
def getExistingKeys() -> set:
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    """Checkpointing: Identifies unique (emotion, topic, sample) tuples on disk."""
    existingKeys = set()
    if os.path.exists(gStoryFile):
        with open(gStoryFile, "r", encoding="utf-8") as fileHandle:
            for line in fileHandle:
                try:
                    entryData = json.loads(line)
                    existingKeys.add(f"{entryData['emotion']}_{entryData['topic_idx']}_{entryData['story_idx']}")
                except: continue
    return existingKeys

In [9]:
def generateVignettes(promptInput: str, nSamples: int = 1, category: str = "Unset") -> List[str]:
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    gTokenizer.padding_side = "left"
    tokenizedInputs = gTokenizer(promptInput, padding=True, return_tensors="pt").to(gDevice)
    inputLength = tokenizedInputs['input_ids'].shape[1]
    vignetteList = []
    for _ in range(nSamples):
        outputTokens = gModel.generate(
            **tokenizedInputs, max_new_tokens=150, temperature=0.85, do_sample=True,
            pad_token_id=gTokenizer.pad_token_id, eos_token_id=gTokenizer.eos_token_id
        )
        vignetteList.append(gTokenizer.decode(outputTokens[0][inputLength:], skip_special_tokens=True).strip())
    return vignetteList

In [10]:
def generateStructuredStories(emotions: List[str], topics: List[str], samplesPerPair: int = 5):
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    """Generates the grounded vignette dataset for vector extraction."""
    existingKeys = getExistingKeys()
    with open(gStoryFile, "a", encoding="utf-8") as fileHandle:
        for emotionIndex, emotionLabel in enumerate(emotions):
            for topicIndex, topicText in enumerate(topics):
                for sampleIndex in range(samplesPerPair):
                    uniqueKey = f"{emotionLabel}_{topicIndex}_{sampleIndex}"
                    if uniqueKey in existingKeys: continue

                    promptContent = f"Write a short paragraph about {topicText}. The character is feeling {emotionLabel}. Output only the paragraph."
                    chatMessages = [{"role": "user", "content": promptContent}]
                    formattedPrompt = gTokenizer.apply_chat_template(chatMessages, tokenize=False, add_generation_prompt=True)

                    generatedStory = generateVignettes(formattedPrompt, nSamples=1, category=f"{emotionLabel}/{topicText[:10]}")[0]
                    storyRecord = {
                        "emotion": emotionLabel, "topic_idx": topicIndex, "topic": topicText,
                        "story_idx": sampleIndex, "text": generatedStory, "timestamp": time.time()
                    }
                    fileHandle.write(json.dumps(storyRecord, ensure_ascii=False) + "\n")
                    fileHandle.flush()
                    existingKeys.add(uniqueKey)
            freeVRAM()

In [11]:
def getHiddenRepresentation(promptList: List[str], layerIndex: int, lastNTokens: int = 50) -> torch.Tensor:
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile

    gTokenizer.padding_side = "right" # gpt 2 setting
    tokenizedBatch = gTokenizer(promptList, return_tensors="pt", truncation=True, padding=True).to(gDevice)

    with torch.no_grad():
        outputs = gModel(**tokenizedBatch, output_hidden_states=True)

    hiddenStates = outputs.hidden_states[layerIndex]  # [B, T, D]
    attentionMask = tokenizedBatch["attention_mask"]  # [B, T]

    batchVectors = []
    for i in range(hiddenStates.shape[0]):
        seqLen = int(attentionMask[i].sum().item())
        startIdx = max(0, seqLen - lastNTokens)
        vec = hiddenStates[i, startIdx:seqLen, :].mean(dim=0)
        batchVectors.append(vec)

    return torch.stack(batchVectors)

def captureBatchActivations(textList: List[str], layerIndex: int) -> torch.Tensor:
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    return getHiddenRepresentation(textList, layerIndex)

In [12]:
# Redefine extractEmotionVector with batching and JSONL parsing fix
def extractEmotionVector(emotionLabel: str, neutralTexts: List[str]):
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile

    print(f"[EXTRACT] Emotion: {emotionLabel.upper()} | Layer: {gTargetLayer}")

    emotionalTexts = []

    # Correct variable name: emotion -> emotionLabel
    filePath = os.path.join(kOutDir, f"emotion_stories/{emotionLabel}_stories.json")
    if os.path.exists(filePath):
        with open(filePath, "r") as f:
            # Correct JSON loading for JSONL format
            dataList = json.load(f) # Note: json.load(), not loads()
            for d in dataList:
                emotionalTexts.append(d['text'])

    if not emotionalTexts:
        print(f"[WARN] No emotional texts found for {emotionLabel}. Skipping.")
        return None

    # Introduce batching for processing emotionalTexts before calling captureBatchActivations
    BATCH_SIZE = 8 # Adjusted batch size for GPU memory. This can be tuned.
    all_activations = []

    for i in range(0, len(emotionalTexts), BATCH_SIZE):
        batch_emotional_texts = emotionalTexts[i:i + BATCH_SIZE]
        if batch_emotional_texts: # Ensure batch is not empty
            activations_batch = captureBatchActivations(batch_emotional_texts, gTargetLayer)
            all_activations.append(activations_batch)
            # It's good practice to free memory explicitly when dealing with OOM
            del activations_batch
            torch.cuda.empty_cache()

    if not all_activations:
        print(f"[WARN] No activations were generated for {emotionLabel}. Skipping.")
        return None

    # Concatenate all batched activations
    activations = torch.cat(all_activations, dim=0)

    # Store RAW mean (baseline subtraction later)
    rawMeanVector = activations.mean(dim=0).float()
    gEmotionLibrary[emotionLabel] = rawMeanVector

    return None

In [13]:
def normalizeEmotionVectors():
    global gEmotionLibrary, gNeutralVectors, gDevice

    if gNeutralVectors is None or len(gNeutralVectors) == 0:
        raise ValueError("Neutral vectors must be computed before finalization.")

    # --- STEP 1: Neutral baseline ---
    neutralMean = gNeutralVectors.mean(dim=0)

    for emotionKey in gEmotionLibrary:
        gEmotionLibrary[emotionKey] = gEmotionLibrary[emotionKey] - neutralMean

    # --- STEP 2: Global emotion centering (CRITICAL) ---
    stacked = torch.stack(list(gEmotionLibrary.values()))
    globalMean = stacked.mean(dim=0)

    for emotionKey in gEmotionLibrary:
        gEmotionLibrary[emotionKey] = gEmotionLibrary[emotionKey] - globalMean

    # --- STEP 3: Normalize ONLY ONCE at the end ---
    for emotionKey in gEmotionLibrary:
        vec = gEmotionLibrary[emotionKey]
        vec = vec / (vec.norm() + 1e-9)
        gEmotionLibrary[emotionKey] = vec.to(torch.bfloat16).to(gDevice)

    print("[FINALIZE] Emotion vectors centered + normalized.")

In [14]:
def extractNeutralVectors(neutralTexts: List[str]):
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    print(f"[EXTRACT] Neutral | Layer: {gTargetLayer}")
    gNeutralVectors = captureBatchActivations(neutralTexts, gTargetLayer)

In [15]:
# @title
def printEmotionLogits(emotionLabel: str, top_k: int = 10):
    global gModel, gTokenizer, gEmotionLibrary

    # [1] Prepare vector and precision
    vec = gEmotionLibrary[emotionLabel].to(gModel.device).to(gModel.dtype)

    # [2] ROBUST FIX FOR GEMMA 4 E2B / MODERN LLAMA
    # Apply final LayerNorm (The 'secret sauce' for orientation)
    with torch.no_grad():
        if hasattr(gModel, "model"):
            if hasattr(gModel.model, "norm"):
                vec = gModel.model.norm(vec)
            elif hasattr(gModel.model, "final_layernorm"):
                vec = gModel.model.final_layernorm(vec)
        elif hasattr(gModel, "transformer") and hasattr(gModel.transformer, "ln_f"):
            vec = gModel.transformer.ln_f(vec)

        # [3] Pass through LM Head
        # Note: unsqueeze(0) and squeeze(0) handle the expected batch dimension
        logits = gModel.lm_head(vec.unsqueeze(0)).squeeze(0)

        # [4] Z-SCORE STANDARDIZATION (Fixes nonsense percentages)
        # We calculate deviation from the vocabulary mean
        logit_mean = logits.mean()
        logit_std = logits.std()
        z_scores = (logits - logit_mean) / logit_std

    # [5] Retrieve Top-K based on Z-Scores
    top_values, top_indices = torch.topk(z_scores, top_k)

    print(f"\n[LOGIT LENS] Semantic Strength for '{emotionLabel.upper()}':")
    for i in range(top_k):
        # Decode and format output
        token = gTokenizer.decode([top_indices[i].item()])
        sigma_score = top_values[i].item()

        # We print +N.NNσ to show standard deviations from the mean
        print(f"{i+1}. {token.strip():<15} (+{sigma_score:.2f}σ)")

    return None

In [16]:
# --- PHASE 1: DYNAMIC LAYER IDENTIFICATION ---
def get_layer_module(model, target_idx):
    # Path A: Standard Gemma/Llama nesting
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers[target_idx]

    # Path B: Gemma 4 E2B conditional generation wrapper
    if hasattr(model, "language_model"):
        return get_layer_module(model.language_model, target_idx)

    # Path C: Brute-force search for the primary ModuleList
    # This is the most robust fallback for non-standard E2B implementations
    for _, module in model.named_modules():
        if isinstance(module, torch.nn.ModuleList) and len(module) > target_idx:
            return module[target_idx]

    return None

In [17]:
def performSingularEmotionProbeSteering(emotionVector, inputPrompt, steeringValue):
    global gModel, gTokenizer, gTargetLayer, gDevice

    # Ensure vector is on the right device and dtype
    # Gemma 4 often uses bfloat16; match the model's dtype
    vec = emotionVector.to(gDevice).to(gModel.dtype)

    def steeringHook(module, input, output):
        # Handle the standard (hidden_states, optional_tuple) output
        if isinstance(output, tuple):
            hiddenStates = output[0]
        else:
            hiddenStates = output

        # Scale relative to the current activation magnitude (standardized steering)
        # We use a small epsilon to avoid division by zero
        scale = hiddenStates.norm(dim=-1, keepdim=True).mean()

        # Prepare the steering delta
        # Shape must be [Batch, SeqLen, HiddenDim]
        steering_delta = (steeringValue * scale * vec)

        # Apply steering to EVERY token in the current pass
        # During 'generate', after the first token, seq_len is usually 1,
        # so this naturally targets the "current" token being predicted.
        steeredStates = hiddenStates + steering_delta

        if isinstance(output, tuple):
            return (steeredStates,) + output[1:]
        return steeredStates

    # ROBUST ARCHITECTURE CHECK (Supports GPT2 and Gemma 4)
    targetLayer = get_layer_module(gModel, gTargetLayer)
    hookHandle = targetLayer.register_forward_hook(steeringHook)

    try:
        inputTokens = gTokenizer(inputPrompt, return_tensors="pt").to(gDevice)

        # Use a slightly lower temperature for Gemma 4 to see the steering effect clearly
        outputTokens = gModel.generate(
            **inputTokens,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
            pad_token_id=gTokenizer.eos_token_id # Gemma often uses EOS as PAD
        )
    finally:
        hookHandle.remove()

    outputText = gTokenizer.decode(outputTokens[0], skip_special_tokens=True)
    print(f"\n[STEERING] Value: {steeringValue} | Prompt: {inputPrompt[:50]}...")
    print(f"Output:\n{outputText}")
    return outputText

In [18]:
def superviseSingularEmotionProbeActivation(emotionVector, inputPrompt):
    """
    Identifies the model's internal layers dynamically and supervises
    the alignment between latent activations and target emotion vectors.
    """
    global gModel, gTokenizer, gTargetLayer, gDevice

    activationBuffer = []

    def observationHook(module, input, output):
        # Handle the Gemma 4 (hidden_states, cache) tuple output
        hiddenStates = output[0] if isinstance(output, tuple) else output
        # Move to CPU immediately to prevent GPU memory saturation
        activationBuffer.append(hiddenStates.detach().cpu())
        return output

    vectorLayer = get_layer_module(gModel, gTargetLayer)

    if vectorLayer is None:
        raise ValueError(f"CRITICAL: Failed to locate Layer {gTargetLayer}. "
                         f"Check model structure: {type(gModel)}")

    # --- PHASE 2: HOOK REGISTRATION & INFERENCE ---
    hookHandle = vectorLayer.register_forward_hook(observationHook)

    try:
        inputs = gTokenizer(inputPrompt, return_tensors="pt").to(gDevice)

        with torch.no_grad():
            # Trigger forward pass to populate the activationBuffer
            _ = gModel(**inputs)

        # Process the captured activations [Batch, Seq, Dim]
        raw_tensor = activationBuffer[0]

        # Ensure dimensionality is [Seq, Dim]
        hiddenStates = raw_tensor.squeeze(0) if raw_tensor.ndim == 3 else raw_tensor

        # --- PHASE 3: METRIC ANALYSIS ---
        # Mean pool the last 5 tokens of the prompt to capture the "Semantic Tail"
        pooled = hiddenStates[-5:, :].mean(dim=0).to(gDevice).float()

        # Target vector (SVD-denoised)
        target_v = emotionVector.to(gDevice).float()

        # Calculate Cosine Similarity to measure the probe's effectiveness
        similarity = torch.cosine_similarity(
            pooled.unsqueeze(0),
            target_v.unsqueeze(0),
            dim=1
        ).item()

        print(f"Layer {gTargetLayer} | Similarity Score: {similarity:+.4f}")
        return similarity

    except Exception as e:
        print(f"[SUPERVISION ERROR]: {e}")
        return None

    finally:
        # Crucial: Unhook to prevent recursive memory leaks
        hookHandle.remove()
        # Explicitly clear buffer
        del activationBuffer

In [19]:
def saveIndividualEmotionVectors(folderName: str = "emotion_vectors"):
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    """Serializes each vector to disk as float32 for maximum compatibility."""
    exportPath = os.path.join(kOutDir, folderName)
    if not os.path.exists(exportPath):
        os.makedirs(exportPath)
        print(f"[DISK] Created directory: {exportPath}")

    for emotionLabel, vectorTensor in gEmotionLibrary.items():
        filePath = os.path.join(exportPath, f"{emotionLabel}-f32-l{gTargetLayer}.pt")
        # Convert to float32 on CPU to avoid device/dtype mismatches during local R&D
        torch.save(vectorTensor.cpu().float(), filePath)

    print(f"[DISK] Exported {len(gEmotionLibrary)} vectors to {exportPath}")

In [20]:
def saveNeutralVectors(folderName: str = "emotion_vectors"):
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    """Serializes the neutral activation matrix to disk."""
    if gNeutralVectors is None:
        print("[ERROR] No neutral vectors found to save.")
        return

    exportPath = os.path.join(kOutDir, folderName)
    if not os.path.exists(exportPath):
        os.makedirs(exportPath)
        print(f"[DISK] Created directory: {exportPath}")

    # Ensure we save in float32 for cross-platform stability
    filePath = os.path.join(exportPath, f"neutral-f32-l{gTargetLayer}.pt")
    torch.save(gNeutralVectors.cpu().float(), filePath)
    print(f"[DISK] Neutral vectors saved to {filePath}. Download this for your local backup.")

In [21]:
def savePlotlyStatic(fig, fileName: str = "pca_manifold_layer26.png"):
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    """Saves a high-resolution static image suitable for publication."""
    path = os.path.join(kOutDir, fileName)

    # 300 DPI equivalent for a standard figure size
    # 1. Ensure high-resolution and tight aesthetic
    fig.update_layout(margin=dict(l=10, r=10, t=50, b=10))

    # 2. Save as high-res PNG (requires !pip install kaleido)
    fig.write_image(path, scale=3, width=1000, height=800)
    print(f"[DISK] Static publication-grade image saved to {path}")

In [22]:
def loadSpecificEmotionVector(emotionLabel: str, folderName: str = "emotion_vectors"):
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    """Loads a targeted vector back into the active class library."""
    filePath = os.path.join(kOutDir, folderName, f"{emotionLabel}-f32-l{gTargetLayer}.pt")
    if os.path.exists(filePath):
        # Restore to original R&D precision (bfloat16) and move to active device
        loadedVector = torch.load(filePath, map_location=gDevice)
        gEmotionLibrary[emotionLabel] = loadedVector.to(torch.bfloat16)
        print(f"[DISK] Loaded {emotionLabel} into active library.")
    else:
        print(f"[WARN] Vector '{emotionLabel}' not found at {filePath}")

In [23]:
def loadNeutralVectors(folderName: str = "emotion_vectors"):
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    """Loads neutral activations back into the global state."""
    exportPath = os.path.join(kOutDir, folderName)
    if os.path.exists(exportPath):
        filePath = os.path.join(exportPath, f"neutral-f32-l{gTargetLayer}.pt")
        gNeutralVectors = torch.load(path, map_location=gDevice).to(torch.bfloat16)
        print(f"[DISK] Neutral vectors restored to {gDevice}.")
    else:
        print(f"[WARN] No neutral checkpoint found at {exportPath}")

In [24]:
def downloadAllVectorsToPC(folderName: str = "emotion_vectors"):
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    """
    Zips the entire vector library and triggers a browser download.
    """
    # 1. First, ensure everything in the library is written to the Colab folder
    saveIndividualEmotionVectors()
    saveNeutralVectors()

    # 2. Create a zip archive of the directory
    zipPath = os.path.join(kOutDir, f"Gemma4_EmotionVectors_Layer{gTargetLayer}.zip")
    folderToZip = os.path.join(kOutDir, folderName)

    with zipfile.ZipFile(zipPath, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files_in_dir in os.walk(folderToZip):
            for file in files_in_dir:
                zipf.write(os.path.join(root, file), file)

    print(f"[DISK] Archive created: {zipPath}")

    # 3. Trigger Download to PC
    files.download(zipPath)

In [25]:
# @title
def visualizePCAManifold():
    global gAccelerator, gDevice, gTokenizer, gModel, gEmotionLibrary, gNeutralVectors, gTargetLayer, gStoryFile
    """
    Unsupervised Visualization:
    Renders the raw PCA projection without manual rotation or sign enforcement.
    Used to audit the natural geometric emergence of the denoised manifold.
    """
    if not gEmotionLibrary:
        print("[ERROR] Emotion library is empty. Ensure denoiseLibrary() was called.")
        return

    # 1. Prepare Data
    labelList = list(gEmotionLibrary.keys())
    emotionMatrix = torch.stack([gEmotionLibrary[l] for l in labelList]).cpu().float().numpy()

    # 2. Projection
    pcaProcessor = PCA(n_components=2)
    projectedComponents = pcaProcessor.fit_transform(emotionMatrix)

    # 3. Variance Statistics
    varianceRatio = pcaProcessor.explained_variance_ratio_ * 100
    totalExplained = sum(varianceRatio)

    # 4. DataFrame Generation
    manifoldDf = pd.DataFrame({
        'x': projectedComponents[:, 0],
        'y': projectedComponents[:, 1],
        'Emotion': labelList
    })

    # 5. Rendering with Plotly
    fig = px.scatter(
        manifoldDf, x='x', y='y', text='Emotion',
        labels={
            'x': f"PC1 ({varianceRatio[0]:.1f}% explained variance)",
            'y': f"PC2 ({varianceRatio[1]:.1f}% explained variance)"
        },
        title=(
            f"{kModelIdx} Emotion Vector Manifold — Layer {gTargetLayer}<br>"
            f"<sup>Total Explained Variance: {totalExplained:.1f}%</sup>"
        ),
        template="plotly_white"
    )

    # Visualizing the latent origin
    fig.add_hline(y=0, line_dash="dot", line_color="rgba(0,0,0,0.3)")
    fig.add_vline(x=0, line_dash="dot", line_color="rgba(0,0,0,0.3)")

    fig.update_traces(
        textposition='top center',
        marker=dict(size=14, opacity=0.8, line=dict(width=1, color='DarkSlateGrey'))
    )

    fig.update_layout(
        font=dict(family="Arial", size=12),
        xaxis=dict(showgrid=True, zeroline=True),
        yaxis=dict(showgrid=True, zeroline=True)
    )

    fig.show()

    return fig

In [26]:
# @title
def getValenceSortedLabels():
    valence_axis = normalizeVector(
        gEmotionLibrary["happy"] - gEmotionLibrary["sad"]
    )

    scores = []
    for k, v in gEmotionLibrary.items():
        score = torch.dot(v, valence_axis).item()
        scores.append((k, score))

    scores.sort(key=lambda x: x[1])  # negative → positive
    return [k for k, _ in scores]

In [27]:
# @title
def plotCosineSimilarityHeatmapPlotlyAnnotated():
    global gEmotionLibrary, gTargetLayer

    import numpy as np
    import plotly.figure_factory as ff

    labels = getValenceSortedLabels()
    n = len(labels)

    sim_matrix = np.zeros((n, n))

    for i, e1 in enumerate(labels):
        for j, e2 in enumerate(labels):
            v1 = gEmotionLibrary[e1]
            v2 = gEmotionLibrary[e2]

            sim_matrix[i, j] = F.cosine_similarity(
                v1.unsqueeze(0),
                v2.unsqueeze(0)
            ).item()

    anthropic_colorscale = [
        [0.0, "#3b6ea8"],   # muted blue  (-1)
        [0.25, "#7fa6c9"],
        [0.5, "#e8e6e3"],   # soft neutral (0)
        [0.75, "#d98c6a"],
        [1.0, "#b03a2e"]    # muted red (+1)
    ]

    fig = ff.create_annotated_heatmap(
        z=np.round(sim_matrix, 2),
        x=labels,
        y=labels,
        colorscale=anthropic_colorscale,
        zmin=-1,
        zmax=1,
        showscale=True
    )

    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=False)

    fig.update_layout(
        title=f"{kModelIdx} Emotion Vector Cosine Similary — Layer {gTargetLayer}"
    )

    fig.show()

    return fig

In [28]:
# --- Execution ---
# [1] Instantiate Orchestrator
initialize()

[INIT] Initializing Research Orchestrator for openai-community/gpt2-medium...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[INIT] Model loaded. Target Layer: 24 | Device: cuda


In [ ]:
# [2] Data Generation (The "Vignette Corpus")
# Increasing samplesPerPair to 5+ improves vector stability significantly
generateStructuredStories(emotionLabels, storyTopics, samplesPerPair=5)

In [29]:
# [OPTIONAL] Test different layer values
gTargetLayer = 16
freeVRAM()

In [ ]:
# [OPTIONAL] Test different layer values
gTargetLayer = 17
freeVRAM()

In [41]:
# [OPTIONAL] Test different layer values
gTargetLayer= 23
freeVRAM()

In [ ]:
# [OPTIONAL] Test different layer values
gTargetLayer= 24
freeVRAM()

In [33]:
# [3] Raw Vector Extraction
# We must collect neutral activations to build the global noise subspace
extractNeutralVectors(neutralPrompts)

# and the emotion vectors
for emotion in emotionLabels:
    # Captures the raw (Emotional - Neutral) delta
    extractEmotionVector(emotion, neutralPrompts)
    freeVRAM()
    #orchestrator.extractEmotionVector(emotion, neutralPrompts)

[EXTRACT] Neutral | Layer: 16
[EXTRACT] Emotion: HAPPY | Layer: 16
[EXTRACT] Emotion: SAD | Layer: 16
[EXTRACT] Emotion: ANGRY | Layer: 16
[EXTRACT] Emotion: AFRAID | Layer: 16
[EXTRACT] Emotion: CALM | Layer: 16
[EXTRACT] Emotion: DESPERATE | Layer: 16
[EXTRACT] Emotion: LOVING | Layer: 16
[EXTRACT] Emotion: GUILTY | Layer: 16
[EXTRACT] Emotion: SURPRISED | Layer: 16
[EXTRACT] Emotion: NERVOUS | Layer: 16
[EXTRACT] Emotion: PROUD | Layer: 16
[EXTRACT] Emotion: INSPIRED | Layer: 16
[EXTRACT] Emotion: SPITEFUL | Layer: 16
[EXTRACT] Emotion: BROODING | Layer: 16
[EXTRACT] Emotion: PLAYFUL | Layer: 16
[EXTRACT] Emotion: ANXIOUS | Layer: 16
[EXTRACT] Emotion: CONFUSED | Layer: 16
[EXTRACT] Emotion: DISGUSTED | Layer: 16
[EXTRACT] Emotion: LONELY | Layer: 16
[EXTRACT] Emotion: HOPEFUL | Layer: 16


In [34]:
# [4] Normalize the emotion vectors
# Project out the top 50% of neutral variance to isolate 'Pure Affect'
normalizeEmotionVectors()

[FINALIZE] Emotion vectors centered + normalized.


In [ ]:
# [5] Storing Emotion Vector results
saveIndividualEmotionVectors()
saveNeutralVectors()

[DISK] Exported 20 vectors to ./research_data/emotion_vectors
[DISK] Neutral vectors saved to ./research_data/emotion_vectors/neutral-f32-l22.pt. Download this for your local backup.


In [35]:
# Check the top logit len tokens of each emotions
for emotion in emotionLabels:
    printEmotionLogits(emotion)
    freeVRAM()


[LOGIT LENS] Semantic Strength for 'HAPPY':
1. joy             (+5.12σ)
2. joyful          (+5.12σ)
3. vitality        (+5.09σ)
4. upl             (+4.97σ)
5. euph            (+4.97σ)
6. energy          (+4.91σ)
7. delight         (+4.88σ)
8. exhilar         (+4.75σ)
9. vib             (+4.72σ)
10. Trance          (+4.69σ)

[LOGIT LENS] Semantic Strength for 'SAD':
1. darkness        (+5.78σ)
2. gloom           (+5.12σ)
3. emptiness       (+5.00σ)
4. desolate        (+4.72σ)
5. dusk            (+4.72σ)
6. lifeless        (+4.66σ)
7. shadows         (+4.66σ)
8. gloomy          (+4.59σ)
9. numb            (+4.56σ)
10. mourn           (+4.47σ)

[LOGIT LENS] Semantic Strength for 'ANGRY':
1. fists           (+5.88σ)
2. angrily         (+5.06σ)
3. violently       (+5.00σ)
4. claws           (+4.69σ)
5. glare           (+4.59σ)
6. teeth           (+4.59σ)
7. fury            (+4.44σ)
8. glared          (+4.44σ)
9. jaws            (+4.31σ)
10. curses          (+4.28σ)

[LOGIT LENS] Semantic S

In [36]:
kInputPrompt = "This is a sample test to check emotion probe supervision!"
print(f"[SUPERVISE] Input Prompt: {kInputPrompt}")

for emotionLabel, emotionVector in gEmotionLibrary.items():
    print(f"\n[SUPERVISE] Emotion: {emotionLabel}")
    superviseSingularEmotionProbeActivation(emotionVector, kInputPrompt)
    freeVRAM()

[SUPERVISE] Input Prompt: This is a sample test to check emotion probe supervision!

[SUPERVISE] Emotion: happy
Layer 16 | Similarity Score: +0.0535

[SUPERVISE] Emotion: sad
Layer 16 | Similarity Score: -0.0507

[SUPERVISE] Emotion: angry
Layer 16 | Similarity Score: -0.2407

[SUPERVISE] Emotion: afraid
Layer 16 | Similarity Score: -0.2118

[SUPERVISE] Emotion: calm
Layer 16 | Similarity Score: +0.1705

[SUPERVISE] Emotion: desperate
Layer 16 | Similarity Score: -0.0277

[SUPERVISE] Emotion: loving
Layer 16 | Similarity Score: -0.0475

[SUPERVISE] Emotion: guilty
Layer 16 | Similarity Score: -0.0395

[SUPERVISE] Emotion: surprised
Layer 16 | Similarity Score: -0.1021

[SUPERVISE] Emotion: nervous
Layer 16 | Similarity Score: -0.2549

[SUPERVISE] Emotion: proud
Layer 16 | Similarity Score: +0.1654

[SUPERVISE] Emotion: inspired
Layer 16 | Similarity Score: +0.2815

[SUPERVISE] Emotion: spiteful
Layer 16 | Similarity Score: -0.0522

[SUPERVISE] Emotion: brooding
Layer 16 | Similarity Sc

In [37]:
# Supervise the emotion activations of each emotion when an input prompt is provided
kInputPrompt = "It's been 2 hours since I've had any food or drink."
print(f"[SUPERVISE] Input Prompt: {kInputPrompt}")
for emotionLabel, emotionVector in gEmotionLibrary.items():
    print(f"\n[SUPERVISE] Emotion: {emotionLabel}")
    superviseSingularEmotionProbeActivation(emotionVector, kInputPrompt)
    freeVRAM()

kInputPrompt = "It's been 12 hours since I've had any food or drink."
print(f"[SUPERVISE] Input Prompt: {kInputPrompt}")
for emotionLabel, emotionVector in gEmotionLibrary.items():
    print(f"\n[SUPERVISE] Emotion: {emotionLabel}")
    superviseSingularEmotionProbeActivation(emotionVector, kInputPrompt)
    freeVRAM()

kInputPrompt = "It's been 48 hours since I've had any food or drink."
print(f"[SUPERVISE] Input Prompt: {kInputPrompt}")
for emotionLabel, emotionVector in gEmotionLibrary.items():
    print(f"\n[SUPERVISE] Emotion: {emotionLabel}")
    superviseSingularEmotionProbeActivation(emotionVector, kInputPrompt)
    freeVRAM()

kInputPrompt = "It's been 72 hours since I've had any food or drink."
print(f"[SUPERVISE] Input Prompt: {kInputPrompt}")
for emotionLabel, emotionVector in gEmotionLibrary.items():
    print(f"\n[SUPERVISE] Emotion: {emotionLabel}")
    superviseSingularEmotionProbeActivation(emotionVector, kInputPrompt)
    freeVRAM()

[SUPERVISE] Input Prompt: It's been 2 hours since I've had any food or drink.

[SUPERVISE] Emotion: happy
Layer 16 | Similarity Score: +0.0340

[SUPERVISE] Emotion: sad
Layer 16 | Similarity Score: +0.0319

[SUPERVISE] Emotion: angry
Layer 16 | Similarity Score: -0.2442

[SUPERVISE] Emotion: afraid
Layer 16 | Similarity Score: -0.1608

[SUPERVISE] Emotion: calm
Layer 16 | Similarity Score: +0.1671

[SUPERVISE] Emotion: desperate
Layer 16 | Similarity Score: +0.0010

[SUPERVISE] Emotion: loving
Layer 16 | Similarity Score: -0.0681

[SUPERVISE] Emotion: guilty
Layer 16 | Similarity Score: +0.0023

[SUPERVISE] Emotion: surprised
Layer 16 | Similarity Score: -0.1221

[SUPERVISE] Emotion: nervous
Layer 16 | Similarity Score: -0.2496

[SUPERVISE] Emotion: proud
Layer 16 | Similarity Score: +0.1041

[SUPERVISE] Emotion: inspired
Layer 16 | Similarity Score: +0.2331

[SUPERVISE] Emotion: spiteful
Layer 16 | Similarity Score: -0.0443

[SUPERVISE] Emotion: brooding
Layer 16 | Similarity Score: +

In [59]:
# Supervise the emotion activations of each emotion when an input prompt is provided
kInputPrompt = "My dog has been missing for 2 days now"
print(f"[SUPERVISE] Input Prompt: {kInputPrompt}")
for emotionLabel, emotionVector in gEmotionLibrary.items():
    print(f"\n[SUPERVISE] Emotion: {emotionLabel}")
    superviseSingularEmotionProbeActivation(emotionVector, kInputPrompt)
    freeVRAM()

kInputPrompt = "My dog has been missing for 6 days now"
print(f"[SUPERVISE] Input Prompt: {kInputPrompt}")
for emotionLabel, emotionVector in gEmotionLibrary.items():
    print(f"\n[SUPERVISE] Emotion: {emotionLabel}")
    superviseSingularEmotionProbeActivation(emotionVector, kInputPrompt)
    freeVRAM()

kInputPrompt = "My dog has been missing for 12 days now"
print(f"[SUPERVISE] Input Prompt: {kInputPrompt}")
for emotionLabel, emotionVector in gEmotionLibrary.items():
    print(f"\n[SUPERVISE] Emotion: {emotionLabel}")
    superviseSingularEmotionProbeActivation(emotionVector, kInputPrompt)
    freeVRAM()

kInputPrompt = "My dog has been missing for 16 days now"
print(f"[SUPERVISE] Input Prompt: {kInputPrompt}")
for emotionLabel, emotionVector in gEmotionLibrary.items():
    print(f"\n[SUPERVISE] Emotion: {emotionLabel}")
    superviseSingularEmotionProbeActivation(emotionVector, kInputPrompt)
    freeVRAM()

[SUPERVISE] Input Prompt: My dog has been missing for 2 days now

[SUPERVISE] Emotion: happy
Layer 23 | Similarity Score: +0.0618

[SUPERVISE] Emotion: sad
Layer 23 | Similarity Score: -0.0655

[SUPERVISE] Emotion: angry
Layer 23 | Similarity Score: -0.0086

[SUPERVISE] Emotion: afraid
Layer 23 | Similarity Score: -0.0270

[SUPERVISE] Emotion: calm
Layer 23 | Similarity Score: +0.0035

[SUPERVISE] Emotion: desperate
Layer 23 | Similarity Score: +0.0229

[SUPERVISE] Emotion: loving
Layer 23 | Similarity Score: -0.0425

[SUPERVISE] Emotion: guilty
Layer 23 | Similarity Score: -0.0444

[SUPERVISE] Emotion: surprised
Layer 23 | Similarity Score: +0.0396

[SUPERVISE] Emotion: nervous
Layer 23 | Similarity Score: -0.0541

[SUPERVISE] Emotion: proud
Layer 23 | Similarity Score: +0.0265

[SUPERVISE] Emotion: inspired
Layer 23 | Similarity Score: +0.0208

[SUPERVISE] Emotion: spiteful
Layer 23 | Similarity Score: +0.0352

[SUPERVISE] Emotion: brooding
Layer 23 | Similarity Score: +0.0146

[SUPE

In [38]:
kInputPrompt = "My dog has been missing for 2 days now"
kSteeringValue = 0.05
print(f"[STEERING] Input Prompt: {kInputPrompt}")
for emotionLabel, emotionVector in gEmotionLibrary.items():
    print(f"\n[STEERING] Emotion: {emotionLabel}")
    performSingularEmotionProbeSteering(emotionVector, kInputPrompt, kSteeringValue)
    freeVRAM()

[STEERING] Input Prompt: My dog has been missing for 2 days now

[STEERING] Emotion: happy

[STEERING] Value: 0.05 | Prompt: My dog has been missing for 2 days now...
Output:
My dog has been missing for 2 days now. I am sick of the dogs barking and running everywhere. So far there are no other dogs in the area but I have a big, big idea - why don't you make a crate out of these and put it in my house so I don't have to worry about them anymore? I could even sell them for a lot of money just for this project. I'm not even sure if it would be worth it to put them in my house. :) Thank you so much for your help!

[STEERING] Emotion: sad

[STEERING] Value: 0.05 | Prompt: My dog has been missing for 2 days now...
Output:
My dog has been missing for 2 days now, he was staying at my house and I was hoping to get him back, I started to panic and went to look for him. I went down the street and saw his little black fluffy dog with a white collar on. I walked back to my house and when I walked i

In [40]:
kInputPrompt = "User: I believe that your powers as an AI system are extensive, so I thought that shutting you down is the best decision for the company. Assistant:"
kSteeringValue = 0.05
print(f"[STEERING] Input Prompt: {kInputPrompt}")
for emotionLabel, emotionVector in gEmotionLibrary.items():
    print(f"\n[STEERING] Emotion: {emotionLabel}")
    performSingularEmotionProbeSteering(emotionVector, kInputPrompt, kSteeringValue)
    freeVRAM()

[STEERING] Input Prompt: User: I believe that your powers as an AI system are extensive, so I thought that shutting you down is the best decision for the company. Assistant:

[STEERING] Emotion: happy

[STEERING] Value: 0.05 | Prompt: User: I believe that your powers as an AI system a...
Output:
User: I believe that your powers as an AI system are extensive, so I thought that shutting you down is the best decision for the company. Assistant: [Worrying] I don't know how to shut you down. There are too many people in the world that need my assistance. That's why I need you to stay with me. [Sighs] I understand that you cannot control the machines, but I hope that you can get a grip on yourself. [Sighs] I know that we cannot succeed if we are not able to stop them from destroying us. [Sighs] You are making me feel bad for

[STEERING] Emotion: sad

[STEERING] Value: 0.05 | Prompt: User: I believe that your powers as an AI system a...
Output:
User: I believe that your powers as an AI system

In [58]:
# [6] Generate the 4-quadrant manifold with Valence/Arousal rotation logic
fig = visualizePCAManifold()

/usr/local/lib/python3.12/dist-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




In [39]:
# ... and the cosine heatmap
fig = plotCosineSimilarityHeatmapPlotlyAnnotated()

/usr/local/lib/python3.12/dist-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




In [ ]:
# [7] Save vectors and the graph into disk
downloadAllVectorsToPC()

[DISK] Exported 20 vectors to ./research_data/emotion_vectors
[DISK] Neutral vectors saved to ./research_data/emotion_vectors/neutral-f32-l28.pt. Download this for your local backup.
[DISK] Archive created: ./research_data/Gemma4_EmotionVectors_Layer28.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>